# The neural network

## Architecture

A (fully connected, feed-forward) neural network is linear regression stacked and bent. Each **node** computes a
weighted sum of its inputs plus a bias, exactly like the straight line of the previous page — and then passes the
result through a non-linear **activation function** $f$:

$$ a = f\Big(\sum_i w_i x_i + b\Big) $$

Nodes are arranged in **layers**; every node of one layer feeds every node of the next. The network we use has the
seven inputs, two **hidden layers** of 32 nodes, and two outputs.

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
from oaisis import data, plots, ml, metrics, selectors, experiments, teaching

In [ ]:
teaching.network_diagram(hidden=(32, 32));

```{figure} images/slide_node.png
:width: 80%
:name: node

Inside one node: inputs, weights, bias, activation.
```

Without the activation function, stacking layers would be pointless — a linear function of a linear function is still
linear, and the whole network would collapse to the straight line of the previous page. With it, the network can
represent curved relationships and interactions between inputs (ice that responds to wind differently when the
concentration is high, say). We use the **ReLU**, $f(z) = \max(0, z)$: it simply switches a node off when its weighted
sum is negative.

In [ ]:
teaching.activations();

## Parameters and hyperparameters

Two very different kinds of numbers describe a network:

| | what | who chooses it |
|---|---|---|
| **parameters** | the weights $w$ and biases $b$ — 1 378 of them here | *the training algorithm*, by minimizing the loss |
| **hyperparameters** | number of layers and nodes, activation function, loss function, learning rate, number of epochs, batch size, regularization | *you*, before training |

The hyperparameters of this tutorial live in one dictionary. The validation set is what you use to compare choices.

In [ ]:
ml.HP

## Learning the parameters: gradient descent

The loss landscape of a network has no closed-form minimum, so it is descended **iteratively**: start from random
weights, compute the gradient of the loss with respect to every parameter, take a step *downhill*, repeat. The step
length is set by the **learning rate**. On the one-weight parabola of the previous page the idea is easy to see — and
so is the one hyperparameter that matters most:

In [ ]:
xs, ys = teaching.xy(n=2000)
teaching.gradient_descent_panel(xs, ys, lrs=(0.1, 0.8, 2.1), steps=15);

Too small a learning rate and the descent crawls; too large and it overshoots and diverges. For a network the
landscape is high-dimensional and bumpy, and the gradient is estimated from a random **batch** of samples at a time
rather than all of them (*stochastic* gradient descent). One pass through the whole training set, batch by batch, is an
**epoch**. The optimizer we use, *Adam*, adapts the step size per parameter, which makes the learning rate much less
delicate than in the picture above — but it is still the first thing to try when training misbehaves.

```{admonition} Try it
:class: tip
In *Live Code* mode call `teaching.gradient_descent(xs, ys, lr=1.0, steps=5)` — or `lr=1.99`, or `w0=0.1`.
```

## Training the network

Now the real thing. `ml.train` builds the network from `ml.HP`, fits it on the training members for `epochs` passes
and evaluates the loss on the validation members after each one.

In [ ]:
p = data.prepared()
model, history = ml.train(p, hp=ml.HP, verbose=0)
model.summary()

## Reading the learning curve

The learning curve shows the loss on the training and validation sets after every epoch. The dashed line is the
validation loss of linear regression, our baseline.

In [ ]:
from sklearn.linear_model import LinearRegression
linear = LinearRegression().fit(p.x_tr, p.y_tr)
baseline = np.mean((linear.predict(p.x_va) - p.y_va) ** 2)

plots.history(history, baseline=baseline);

Three things to read off a learning curve:

* **Is it still going down?** If the validation loss is still falling at the last epoch, train longer. Here most of the
  improvement happens in the first few epochs, because the problem is close to linear and the network finds the line
  almost immediately; the remaining slow decrease is it learning the non-linear corrections.
* **Is it below the baseline?** Yes, by a little — the honest size of the non-linear information in these inputs.
* **Do training and validation agree?** They should, when the model generalises. The validation loss is often *lower*
  than the training loss here; that is normal and comes from the L2 regularization term, which is counted in the
  training loss only.

## Overfitting

When training and validation curves *disagree* — training loss keeps falling while validation loss flattens or
rises — the model is **overfitting**: it is memorising the training samples, noise included, instead of learning
the relationship. It happens when the model has far more freedom than the data constrain. We can make it happen on
purpose: a much bigger network (three layers of 256 nodes, 135 000 parameters), no regularization, small batches, 400 epochs — and only the 62 days of one short ITP track as training data.

In [ ]:
p_small = data.prepared(selectors.tracks(95))                   # one ITP, two months of data
hp_over = dict(ml.HP, n_units=256, n_layers=3, l2=0.0, batch_size=32, epochs=400)   # big, unregularized, long
model_over, history_over = ml.train(p_small, hp=hp_over)
plots.history(history_over, ylim=(0, 0.3));

The training loss keeps falling — the network is learning these 62 × 20 samples by heart — while the validation
loss reaches its minimum within the first few epochs and then climbs steadily. Anything learned after that point is
noise. (Compare with the healthy curve above, where the two lines stay together for 30 epochs.) The remedies are the usual ones:
**more data** (the whole point of adding observations, as the last page shows), **regularization** (our L2 penalty on
the weights; *dropout*, which randomly switches nodes off during training), a **smaller network**, or simply
**stopping early** at the validation minimum.

## Evaluating on unseen data

The validation set was used to make choices, so the final score comes from the **test** members, which the
network has never seen in any way.

In [ ]:
Y_pred = ml.predict(model, p.scaler, p.X_te)
Y_lin = p.scaler.y_inv(linear.predict(p.x_te))

metrics.report(p.Y_te, Y_lin,  label="linear")
metrics.report(p.Y_te, Y_pred, label="network")
plots.prediction_scatter(p.Y_te, Y_pred, title="test members:")
teaching.residual_histogram(p.Y_te, Y_pred, Y_lin);

The network beats the linear baseline by a few hundredths of skill. The scatter is tight for small velocities and
widens for fast ice; the map shows where the error is largest along the tracks — typically at the edges of the gyre
and towards Fram Strait, where the ice moves fastest and the coast matters.

```{admonition} Try it
:class: tip
Everything in `ml.HP` is yours to change in *Live Code* mode: `ml.train(p, hp=dict(ml.HP, n_units=8, n_layers=1))`,
`lr=1e-2`, `loss=metrics.nrmse_loss`, `dropout=0.2`, `activation="tanh"` … Watch the learning curve, then the test score.
```

Next: [from ITP tracks to the whole Arctic](05_reconstruction.ipynb).